In [9]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT 
from psycopg2.extras import execute_batch

import pandas as pd

In [10]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: '))
with open('pwd.txt', 'r') as file:
    pwd = file.read()
conn = psycopg2.connect(
        dbname='postgres',
        user='postgres',
        password=pwd,
        host='localhost',
        port='5432')

In [11]:
cur = conn.cursor()

In [12]:
df = pd.read_csv('flights.csv')

mapping = {'int64' : 'INTEGER',
        'str' : 'VARCHAR(100)',
        'object': 'VARCHAR(100)',
        'float64' : 'NUMERIC(10, 2)',
        'bool' : 'BOOLEAN'}
postgre_dtypes = df.dtypes.reset_index(drop=False).iloc[:, 1].astype(str)
postgre_dtypes = postgre_dtypes.apply(lambda x: mapping[x])

cols = ', '.join([f'"{col}"' for col in df.columns])
columns = ''
for c, t in zip(df.columns, postgre_dtypes):
    columns += f'"{c}" {t}, '
columns = columns.strip(', ')
vals = ', '.join(['%s'] * len(df.columns))

cur.execute(f'''DROP TABLE IF EXISTS data; 
            CREATE TABLE data ({columns});''')
cur.executemany(f'INSERT INTO data ({cols}) VALUES ({vals});', [row for row in df.itertuples(index=False)])

# дату рейса переводим в тип даты
cur.execute('''ALTER TABLE data 
ALTER COLUMN "Departure Date" TYPE TIMESTAMP 
USING TO_TIMESTAMP("Departure Date", 'YYYY-MM-DD HH24:MI:SS');''')

In [ ]:
# создаем "очищенную таблицу", на случай если нет значения в столбце задержки (для использования COALESCE)
cur.execute("""
CREATE TABLE data_clean AS
SELECT *, COALESCE("Delay Minutes", 0) AS "Delay Minutes Clean" FROM data;""")

# создаем таблицу расписания рейсов
cur.execute('''
CREATE TABLE flights_schedule AS
SELECT
    ROW_NUMBER() OVER (ORDER BY "Route", "Departure Date") AS "Flight ID",
    "Route",
    "Departure Date",
    MAX("Delay Minutes Clean") AS "Delay Minutes Clean",
    COUNT(*) AS "Passenger Count",
    AVG("Ticket Price") AS "Avg Ticket Price"
FROM data_clean
GROUP BY "Route", "Departure Date";
''')

# создаем таблицу пассажиры
cur.execute('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT DISTINCT ON ("Customer ID")
    "Customer ID",
    "Name",
    COUNT(*) OVER(PARTITION BY "Customer ID") AS "Flights count",
    "Frequent Flyer Status",
    "Loyalty Points"
FROM data_clean
ORDER BY "Customer ID", "Departure Date" DESC;
''')

In [ ]:
# создаем айди рейсов
cur.execute("""
ALTER TABLE data_clean
ADD COLUMN flight_id INT;
""")
cur.execute("""
WITH ranked AS (
    SELECT
        ctid,
        DENSE_RANK() OVER (
            ORDER BY arrival_city, route, origin, destination, departure_city	
        ) AS new_flight_group_id
    FROM flights_raw
)
UPDATE flights_raw f
SET flight_id = r.new_flight_group_id
FROM ranked r
WHERE f.ctid = r.ctid;
""") 
# with ranked as тут создаем временную таблицу для запроса
# dense_ rank присвоит номер каждой уникальной комбинации (по которым будем группировать строки где совпадают заданные параметры, не уточняю их, на случай если хотим их покрутить разные)
# DATE(departure_date) - тут по дню смотрим (не вплоть до секунд, иначе там вообще все рейсы уникальный айди)
df_sql = pd.read_sql("""
SELECT *
FROM flights_raw
ORDER BY flight_id ASC
""", conn)

df_sql.tail(5)

C:\Users\DELL\AppData\Local\Temp\ipykernel_352\3084550003.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql = pd.read_sql("""


,customer_id,departure_city,arrival_city,departure_date,flight_duration,delay_minutes,name,booking_class,frequent_flyer_status,route,ticket_price,competitor_price,demand,origin,destination,profitability,loyalty_points,churned,flight_id
1145,10036,New Ashley,West Victoriaside,2023-08-02 04:24:04,1.144913,136,Mary Gonzalez,Economy,Silver,MEL-BNE,161.783203,249.797795,-0.740666,MEL,LHR,1.373928,4653,True,101
1146,10631,New Ashley,West Victoriaside,2023-09-18 20:35:33,1.159195,22,Karen Rafaelyan,Business,Gold,MEL-BNE,151.150892,191.551673,-0.770073,MEL,LHR,1.259814,3642,False,101
1147,10680,New Ashley,West Victoriaside,2023-04-30 18:58:58,1.353926,13,John Martin,First,Silver,MEL-BNE,121.526141,206.252715,-1.002130,MEL,LHR,1.221388,388,True,101
1148,6638,New Ashley,West Victoriaside,2023-05-25 09:33:02,1.275810,96,Mary Barber,Economy,Gold,MEL-BNE,141.558089,241.613815,-0.811397,MEL,LHR,1.180048,2662,True,101
1149,9968,New Ashley,West Victoriaside,2023-08-03 03:42:09,1.224756,112,Jennifer Martinez,Economy,Silver,MEL-BNE,151.875060,214.351735,-0.858883,MEL,LHR,1.369932,4812,False,101
